# Lecture 8 — Class Exercise
## Choropleth Maps

> **Push to:** `week08/lecture08_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:
# Task 1
# Load Gapminder data for 2007
gm = px.data.gapminder()
gm_2007 = gm[gm['year'] == 2007].copy()

# Compute each country's deviation from the global mean life expectancy
mean_life_exp = gm_2007['lifeExp'].mean()
gm_2007['lifeExp_diff'] = gm_2007['lifeExp'] - mean_life_exp

print(f"Global average life expectancy in 2007: {mean_life_exp:.1f} years")

# Build choropleth with diverging colour scale centered at zero
fig1 = px.choropleth(
    gm_2007,
    locations='iso_alpha',
    locationmode='ISO-3',
    color='lifeExp_diff',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    hover_name='country',
    hover_data={
        'lifeExp': ':.1f',
        'lifeExp_diff': ':.1f',
        'iso_alpha': False,
        'country': False
    },
    labels={
        'lifeExp': 'Life Expectancy (yrs)',
        'lifeExp_diff': f'Diff vs Avg ({mean_life_exp:.1f} yrs)'
    },
    title='Sub-Saharan Africa falls furthest below the 2007 global average life expectancy (67.0 years)',
    projection='natural earth',
    height=550
)

fig1.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=0, r=0, t=55, b=0),
    coloraxis_colorbar=dict(title='vs Avg<br>(years)', thickness=15, len=0.6),
    geo=dict(showframe=False, showcoastlines=True)
)

fig1.show(renderer="png")


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** PublicaMundi MappingAPI repository on GitHub (`https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json`).

**Chart type chosen** (`px.choropleth` or `px.choropleth_mapbox` / `px.choropleth_map`) **and reason:**

I chose `px.choropleth_map` (the modern interactive tile map successor to `px.choropleth_mapbox` in Plotly 2.35+) because it provides an interactive Mapbox tile basemap. This allows users to dynamically zoom in on specific coastal regions or smaller urban states and inspect underlying terrain and city distributions beneath the shaded state polygons, making exploratory spatial analysis much more interactive and informative than a static projected map.

**Colour scale chosen** (sequential or diverging) **and reason:**

I chose a **sequential** colour scale (`YlOrRd` — Yellow/Orange/Red) because Population Density is a strictly positive, one-directional numerical variable ranging from sparsely inhabited regions to high-density areas. Since there is no natural negative spectrum or meaningful zero-divergence threshold for population density, a sequential scale naturally guides the eye from light hues (low population density) to dark, warm hues (high population concentration).


In [ ]:
# Task 2
import json
import urllib.request
import pandas as pd
import plotly.express as px

# 1. Fetch free online US States GeoJSON
url = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
with urllib.request.urlopen(url) as response:
    us_geojson = json.loads(response.read().decode('utf-8'))

# 2. Inspect properties to verify the matching key for featureidkey
sample_props = us_geojson['features'][0]['properties']
print("Sample GeoJSON properties:", sample_props)
# Expected output: {'name': 'Alabama', 'density': 94.65}
# Hence, featureidkey must be set to 'properties.name'

# 3. Create a matching dataset using the density variable across US states
state_records = []
for feature in us_geojson['features']:
    props = feature['properties']
    state_records.append({
        'State': props['name'],
        'Density': props['density']
    })

df_states = pd.DataFrame(state_records).sort_values(by='Density', ascending=False)
print(df_states.head())

# 4. Build interactive choropleth tile map with a sequential colour scale
fig2 = px.choropleth_map(
    data_frame=df_states,
    geojson=us_geojson,
    locations='State',
    featureidkey='properties.name',
    color='Density',
    color_continuous_scale='YlOrRd',
    range_color=[0, 500],  # Cap range at 500 to prevent ultra-dense anomalies (DC/NJ) from skewing visual variation across typical states
    hover_name='State',
    hover_data={'State': False, 'Density': ':.1f'},
    labels={'Density': 'Pop Density<br>(people/sq mi)'},
    title='US Population Density is heavily concentrated along the North-East Corridor and Coastal States, while Mountain West remains sparsely populated',
    center={'lat': 37.8, 'lon': -96.0},  # Geographic center of continental USA
    zoom=3.2,
    opacity=0.75,
    height=600
)

fig2.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=0, r=0, t=55, b=0)
)

fig2.show(renderer="png")
